In [ ]:
#!kaggle competitions download -c springleaf-marketing-response

In [ ]:
#!dir

In [ ]:
#import zipfile

#with zipfile.ZipFile('springleaf-marketing-response.zip', 'r') as zip_ref:
#    zip_ref.extractall('springleaf_data')


In [ ]:
#!dir

In [ ]:
#!head -50000 train/train.csv > smalltrain.csv
#!head -50000 test/test.csv > smalltest.csv

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [2]:
traindf = pd.read_csv("smalltrain.csv", low_memory=False)
testdf = pd.read_csv("smalltest.csv", low_memory=False)
print(f"Starting shape: {traindf.shape}")

Starting shape: (49999, 1934)


In [4]:
copydf = traindf.copy()
print("done")
#just a backup dataframe to compare to if needed
#traindf=copydf.copy

done


In [5]:
suspect_cols = [8,9,10,11,12,43,157,196,214,225,228,229,231,235,238]
# when running code to read csv it stated these columns contained mixed datatypes
col_names = traindf.columns[suspect_cols]

for col in col_names:
    print(f"--- {col} ---")
    print(traindf[col].apply(type).value_counts(), "\n")

    print(f"Unique values in '{col}':")
    print(traindf[col].unique())
    print("-" * 40)


--- VAR_0008 ---
VAR_0008
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0008':
[False nan]
----------------------------------------
--- VAR_0009 ---
VAR_0009
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0009':
[False nan]
----------------------------------------
--- VAR_0010 ---
VAR_0010
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0010':
[False nan]
----------------------------------------
--- VAR_0011 ---
VAR_0011
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0011':
[False nan]
----------------------------------------
--- VAR_0012 ---
VAR_0012
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0012':
[False nan]
----------------------------------------
--- VAR_0043 ---
VAR_0043
<class 'bool'>     49980
<class 'float'>    

In [7]:
# Get non-numeric (categorical) columns
non_numeric_cols = traindf.select_dtypes(exclude=['number']).columns.tolist()

# Count of categorical columns
categorical_count = len(non_numeric_cols)

print(f"Non-numeric (categorical) columns ({categorical_count} total):")
print(non_numeric_cols)

Non-numeric (categorical) columns (51 total):
['VAR_0001', 'VAR_0005', 'VAR_0008', 'VAR_0009', 'VAR_0010', 'VAR_0011', 'VAR_0012', 'VAR_0043', 'VAR_0044', 'VAR_0073', 'VAR_0075', 'VAR_0156', 'VAR_0157', 'VAR_0158', 'VAR_0159', 'VAR_0166', 'VAR_0167', 'VAR_0168', 'VAR_0169', 'VAR_0176', 'VAR_0177', 'VAR_0178', 'VAR_0179', 'VAR_0196', 'VAR_0200', 'VAR_0202', 'VAR_0204', 'VAR_0214', 'VAR_0216', 'VAR_0217', 'VAR_0222', 'VAR_0226', 'VAR_0229', 'VAR_0230', 'VAR_0232', 'VAR_0236', 'VAR_0237', 'VAR_0239', 'VAR_0274', 'VAR_0283', 'VAR_0305', 'VAR_0325', 'VAR_0342', 'VAR_0352', 'VAR_0353', 'VAR_0354', 'VAR_0404', 'VAR_0466', 'VAR_0467', 'VAR_0493', 'VAR_1934']


In [8]:
# above columns seem to be primarily boolean, let's see which are likely all boolean
# Coerce columns with mostly boolean-like data and then we'll look at the two remaining suspicious columns
for col in col_names:
    if col not in ["VAR_0157", "VAR_0214"]:
        traindf[col] = traindf[col].astype(str).map(lambda x: x.strip() in ['1', 'True', 'Y', 'Yes'])


In [9]:
boolean_like = {True, False, 1, 0, '1', '0', 'True', 'False', 'Y', 'N', 'Yes', 'No'}
# Loop through suspect columns and print unusual values
for col in col_names:
    unique_vals = set(traindf[col].dropna().unique())
    unusual_vals = unique_vals - boolean_like
    if unusual_vals:
        print(f"Column {col} has unusual values: {unusual_vals}\n")

Column VAR_0157 has unusual values: {'27JAN12:00:00:00', '12FEB12:00:00:00', '19MAR12:00:00:00', '27OCT11:00:00:00', '13JUN10:00:00:00', '18SEP12:00:00:00', '24JUN12:00:00:00', '23AUG12:00:00:00', '24SEP12:00:00:00', '21AUG11:00:00:00', '06OCT11:00:00:00', '13JUN12:00:00:00', '29OCT11:00:00:00', '30MAR12:00:00:00', '26DEC11:00:00:00', '27DEC11:00:00:00', '17MAR12:00:00:00', '10MAR12:00:00:00', '16NOV11:00:00:00', '31JUL12:00:00:00', '02JUN12:00:00:00', '12JUL11:00:00:00', '24APR12:00:00:00', '20SEP12:00:00:00', '21MAR12:00:00:00', '07MAY10:00:00:00', '17APR12:00:00:00', '30JAN10:00:00:00', '22OCT11:00:00:00', '13DEC11:00:00:00', '26AUG12:00:00:00', '01SEP11:00:00:00', '29DEC11:00:00:00', '23MAY11:00:00:00', '29SEP11:00:00:00', '03JAN11:00:00:00', '09MAR12:00:00:00', '13NOV11:00:00:00', '22DEC11:00:00:00', '05JUN10:00:00:00', '09JAN11:00:00:00', '24JUL12:00:00:00', '10NOV11:00:00:00', '14MAR12:00:00:00', '30NOV11:00:00:00', '13JUL12:00:00:00', '21JUL12:00:00:00', '13JAN12:00:00:00', '12

In [10]:
for col in col_names:
    print(f"--- {col} ---")
    print(traindf[col].apply(type).value_counts(), "\n")

    print(f"Unique values in '{col}':")
    print(traindf[col].unique())
    print("-" * 40)

--- VAR_0008 ---
VAR_0008
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0008':
[False]
----------------------------------------
--- VAR_0009 ---
VAR_0009
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0009':
[False]
----------------------------------------
--- VAR_0010 ---
VAR_0010
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0010':
[False]
----------------------------------------
--- VAR_0011 ---
VAR_0011
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0011':
[False]
----------------------------------------
--- VAR_0012 ---
VAR_0012
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0012':
[False]
----------------------------------------
--- VAR_0043 ---
VAR_0043
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0043':
[False]
----------------------------------------
--- VAR_0157 ---
VAR_0157
<class 'float'>    49673
<class 'str'>

In [11]:
missing_pct = traindf.isnull().mean()
high_missing = missing_pct[missing_pct > 0.90].index
traindf.drop(columns=high_missing, inplace=True)
print(f"Dropped {len(high_missing)} high-missing columns\n{high_missing}")

Dropped 17 high-missing columns
Index(['VAR_0156', 'VAR_0157', 'VAR_0158', 'VAR_0159', 'VAR_0166', 'VAR_0167',
       'VAR_0168', 'VAR_0169', 'VAR_0177', 'VAR_0178', 'VAR_0205', 'VAR_0206',
       'VAR_0207', 'VAR_0209', 'VAR_0213', 'VAR_0214', 'VAR_0840'],
      dtype='object')


In [12]:
#looks like those two problematic columns had less than 10% data filled anyway, as they've been dropped we move on
nunique = traindf.nunique()
constant_cols = nunique[nunique == 1].index
traindf.drop(columns=constant_cols, inplace=True)
print(f"Dropped {len(constant_cols)} columns which only have one value")

Dropped 54 columns which only have one value


In [13]:
# Gets a list of duplicate columns
def get_duplicate_columns(df):
    duplicates = set()
    cols = df.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if cols[j] not in duplicates and df[cols[i]].equals(df[cols[j]]):
                duplicates.add(cols[j])
    return list(duplicates)

# Drops those columns
dupe_cols = get_duplicate_columns(traindf)
traindf.drop(columns=dupe_cols, inplace=True)

print(f"Dropped {len(dupe_cols)} duplicate columns: {dupe_cols}")

Dropped 13 duplicate columns: ['VAR_0357', 'VAR_0201', 'VAR_0228', 'VAR_0013', 'VAR_0238', 'VAR_0182', 'VAR_0512', 'VAR_0130', 'VAR_0398', 'VAR_0211', 'VAR_0181', 'VAR_1036', 'VAR_0210']


In [14]:
print(f"Remaining shape: {traindf.shape}")

Remaining shape: (49999, 1850)


In [15]:
# Get non-numeric (categorical) columns
non_numeric_cols = traindf.select_dtypes(exclude=['number']).columns.tolist()

# Count of categorical columns
categorical_count = len(non_numeric_cols)

print(f"Non-numeric (categorical) columns ({categorical_count} total):")
print(non_numeric_cols)

Non-numeric (categorical) columns (27 total):
['VAR_0001', 'VAR_0005', 'VAR_0073', 'VAR_0075', 'VAR_0176', 'VAR_0179', 'VAR_0200', 'VAR_0204', 'VAR_0217', 'VAR_0226', 'VAR_0230', 'VAR_0232', 'VAR_0236', 'VAR_0237', 'VAR_0274', 'VAR_0283', 'VAR_0305', 'VAR_0325', 'VAR_0342', 'VAR_0352', 'VAR_0353', 'VAR_0354', 'VAR_0404', 'VAR_0466', 'VAR_0467', 'VAR_0493', 'VAR_1934']


In [16]:
for col in non_numeric_cols:
    print(f"\nColumn: {col}")
    print("-" * (len(col) + 9))
    print(traindf[col].value_counts(dropna=False).head(30))


Column: VAR_0001
-----------------
VAR_0001
R    29280
H    20532
Q      187
Name: count, dtype: int64

Column: VAR_0005
-----------------
VAR_0005
B    24747
C    18514
N     5727
S     1011
Name: count, dtype: int64

Column: VAR_0073
-----------------
VAR_0073
NaN                 34777
13MAR09:00:00:00       99
12MAR09:00:00:00       71
20SEP11:00:00:00       55
03OCT11:00:00:00       53
18MAY12:00:00:00       52
19JUL12:00:00:00       52
14MAY12:00:00:00       51
13DEC11:00:00:00       50
10JUN11:00:00:00       49
30MAY12:00:00:00       49
03AUG12:00:00:00       48
30AUG12:00:00:00       46
22MAY12:00:00:00       45
07AUG12:00:00:00       45
18JUL12:00:00:00       45
26JUN12:00:00:00       45
08MAY12:00:00:00       45
04JUN12:00:00:00       45
07JUN12:00:00:00       45
08AUG12:00:00:00       44
09APR12:00:00:00       44
04SEP12:00:00:00       44
01MAY12:00:00:00       44
21MAY12:00:00:00       44
05SEP12:00:00:00       43
18JUN12:00:00:00       43
03MAY12:00:00:00       42
14DEC11:

In [20]:
# Looks like multiple boolean columns are stored as categorical still, so let's address those.

suspect_cols2 = ["VAR_0226", "VAR_0230", "VAR_0232", "VAR_0236"]
print(traindf[suspect_cols2].dtypes)
for col in suspect_cols2:
    unique_vals = set(traindf[col].dropna().unique())
    unusual_vals = unique_vals - boolean_like
    if unusual_vals:
        print(f"Column {col} has unusual values: {unusual_vals}\n")


In [24]:
# There's so many columns, for now I'll only look at numeric columns for correlation to target
numeric_cols = traindf.select_dtypes(include=['number']).columns

correlations = traindf[numeric_cols].corrwith(traindf['target']).abs()

top_corr = correlations.sort_values(ascending=False)
print(top_corr.head(20))

target      1.000000
VAR_0105    0.210117
VAR_0505    0.207853
VAR_0121    0.205860
VAR_0104    0.203623
VAR_0120    0.200894
VAR_0886    0.200230
VAR_0145    0.196881
VAR_0113    0.196856
VAR_0103    0.193631
VAR_0119    0.193221
VAR_0015    0.192538
VAR_0144    0.192191
VAR_0017    0.191604
VAR_0137    0.190103
VAR_0006    0.188191
VAR_0795    0.186871
VAR_0503    0.186260
VAR_0143    0.185884
VAR_0112    0.182003
dtype: float64


In [25]:
#Now I look at the correlation to each other
numeric_df = traindf.select_dtypes(include=[np.number])

corr_matrix = numeric_df.corr().abs()  # Use absolute value to ignore sign

# Fill diagonal with NaN so self-correlation doesn't affect the average
np.fill_diagonal(corr_matrix.values, np.nan)

# Compute average correlation for each column
avg_corr = corr_matrix.mean()

most_unique_cols = avg_corr.sort_values().index[:50]
print("Top 50 most unique columns based on average correlation:")
print(most_unique_cols.tolist())


Top 50 most unique columns based on average correlation:
['VAR_0411', 'VAR_0463', 'VAR_0191', 'VAR_0449', 'VAR_0395', 'VAR_0340', 'VAR_0180', 'VAR_0459', 'ID', 'VAR_0445', 'VAR_0192', 'VAR_0396', 'VAR_1427', 'VAR_0521', 'VAR_0437', 'VAR_0270', 'VAR_0074', 'VAR_0386', 'VAR_0909', 'VAR_0310', 'VAR_0397', 'VAR_0193', 'VAR_0333', 'VAR_0502', 'VAR_0330', 'VAR_0331', 'VAR_0303', 'VAR_0243', 'VAR_0289', 'VAR_0264', 'VAR_0194', 'VAR_0399', 'VAR_0436', 'VAR_0107', 'VAR_0098', 'VAR_0195', 'VAR_0138', 'VAR_0443', 'VAR_0377', 'VAR_0551', 'VAR_0387', 'VAR_0524', 'VAR_0114', 'VAR_0496', 'VAR_0393', 'VAR_0315', 'VAR_0371', 'VAR_0428', 'VAR_0388', 'VAR_0284']


In [ ]:
# Get top 100 most unique columns (lowest average correlation to others)
top_unique_cols = avg_corr.sort_values().index[:100]

# Get top 100 most target-correlated columns (highest abs correlation to target)
top_target_corr_cols = correlations.sort_values(ascending=False).index[:100]

# Find intersection (columns in both lists)
best_of_both = list(set(top_unique_cols) & set(top_target_corr_cols))

print(f"Found {len(best_of_both)} columns that are both unique and correlated with target:")
print(best_of_both)